# TrakrAI GPU worker for Google Colab

This notebook lends a Colab GPU to **your factory's** model training in TrakrAI. It only runs jobs for your factory, and it stops when you stop this notebook.

**Run it**

1. In TrakrAI, open **Model training → Compute** and click **Connect a Colab GPU**. It shows a pairing code and these same steps with pictures.
2. Paste the code into the `PAIRING_CODE` box below.
3. Click **Run all** in the toolbar (or **Runtime → Run all**). If Colab warns that the notebook was not authored by Google, choose **Run anyway**.

The cell keeps running while the worker is connected; its log appears underneath. A pairing code works once and expires after 15 minutes; create a new one in TrakrAI if it has expired.

A code created from a job's GPU worker list (**Start a Colab GPU for this job only**) connects this runtime for that one job: the worker exits when the job finishes, or after 30 minutes without work, and the runtime releases itself so no more compute units are used.

**Before you start**

- This notebook asks Colab for a **T4 GPU** when it opens; the top-right corner shows **T4**. If it shows something else, choose **Runtime → Change runtime type → T4 GPU** and **Save**.
- Use a Colab account with a paid plan or a positive compute-unit balance. Colab's terms do not allow running workers like this one on the free tier.

**Stop it**

Press the stop button on the cell, or **Runtime → Disconnect and delete runtime**. A job that was running is handed back to TrakrAI and picked up again later. Your compute-unit usage is shown under **Runtime → View resources** and in Colab's billing page; TrakrAI shows which jobs this worker ran and for how long.


In [ ]:
# @title Connect this runtime to TrakrAI
TRAKRAI_URL = "https://ai.trakr.live"
PAIRING_CODE = ""  # @param {type:"string"}
JOB_SLOTS = "train_detector:1,extract_frames:1,pretag_batch:1"  # @param {type:"string"}

import io
import json
import os
import shutil
import sys
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path(os.environ.get("TRAKRAI_COLAB_ROOT", "/content/trakrai"))
PKG = ROOT / "pkg"
STATE = ROOT / "worker.json"
ROOT.mkdir(parents=True, exist_ok=True)

if shutil.which("nvidia-smi") is None and not os.environ.get("TRAKRAI_ALLOW_NO_GPU"):
    raise SystemExit(
        "No GPU on this runtime. Choose Runtime > Change runtime type > T4 GPU and run again."
    )


def call(method, path, body=None, headers=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(
        TRAKRAI_URL + path,
        data=data,
        headers={"Content-Type": "application/json", **(headers or {})},
        method=method,
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read() or b"{}")
    except urllib.error.HTTPError as error:
        detail = error.read().decode(errors="replace")
        try:
            detail = json.loads(detail).get("error", detail)
        except ValueError:
            pass
        raise SystemExit(f"TrakrAI said: {detail}") from None


def pair_with_code(code):
    return call("POST", "/api/ml-worker/pair", {"code": code, "hostname": os.uname().nodename})


saved = json.loads(STATE.read_text()) if STATE.exists() else {}
token = saved.get("token")
single_job = bool(saved.get("singleJob"))
if token is None:
    if not PAIRING_CODE.strip():
        raise SystemExit(
            "Paste a pairing code into PAIRING_CODE. Create one in TrakrAI: Model training > "
            "Compute > Connect a Colab GPU."
        )
    paired = pair_with_code(PAIRING_CODE.strip())
    token = paired["token"]
    single_job = bool(paired.get("singleJob"))
    STATE.write_text(json.dumps({"token": token, "name": paired["name"], "singleJob": single_job}))
    STATE.chmod(0o600)
    print(f"Paired as {paired['name']} for {paired['factoryName']}.", flush=True)
    if single_job:
        print("This runtime will run one job, then stop and release itself.", flush=True)

request = urllib.request.Request(
    TRAKRAI_URL + "/api/ml-worker/package", headers={"Authorization": "Bearer " + token}
)
try:
    with urllib.request.urlopen(request, timeout=300) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
except urllib.error.HTTPError:
    STATE.unlink(missing_ok=True)
    raise SystemExit(
        "This worker's token is no longer valid. Run the cell again to connect it afresh."
    ) from None
shutil.rmtree(PKG, ignore_errors=True)
try:
    archive.extractall(PKG, filter="data")
except TypeError:
    archive.extractall(PKG)
sys.path.insert(0, str(PKG))

from trakrai_ml_worker.colab import run_worker

exit_code = run_worker(
    TRAKRAI_URL,
    token,
    PKG,
    slots=JOB_SLOTS,
    exit_after_jobs=1 if single_job else 0,
    idle_exit_seconds=1800 if single_job else 0,
    workdir=str(ROOT / "work"),
)
print(f"Worker stopped (exit code {exit_code}).")
if exit_code == 3 or single_job:
    STATE.unlink(missing_ok=True)
if single_job:
    try:
        from google.colab import runtime

        print("Releasing this Colab runtime.", flush=True)
        runtime.unassign()
    except ImportError:
        pass